In [ ]:
#READS AN EXCEL FILE WITH ALL DATASETS USED IN EACH PUBLICATION (EXTRACTED FROM HORT'S SURVEY)
#STARTING FROM A N EXISTING ONTOLOGY papers.ttl POPULATES ANOTHER ONTOLOGY IN papers_upd1.ttl FILE WITH ALL THESE ENTRIES (PAPERS LINKED TO DATASETS)
#POPULATES THE Datasets and ApplicationDomain SHEET OF Fairness-Metrics.xlsx WITH NEWLY FOUND DATASETS AND THE PAPERS CITING THEM


from rdflib import Graph
import pandas as pd
from rdflib import URIRef,Literal, Namespace 
from rdflib.namespace import RDF,RDFS,split_uri

In [71]:
# Creating a copy of a given ttl file
g = Graph()
g_new=Graph()
g.parse("./complete-ontologies/v4/papers.ttl", format='turtle')
print(len(g))


34924


In [ ]:
df = pd.read_excel("/Users/daniela/Desktop/HORT.xlsx", sheet_name="Foglio1",header=0, dtype=str)
df.fillna('', inplace=True)

df.head()


def camelCaseString(s):  
    temp = s.replace('_', ' ').replace('-', ' ').split('(')[0]
    temp = ' '.join([w.title() if w.islower() else w for w in temp.split()])
    temp=temp.replace(' ', '')
    #res = temp[0].lower() + temp[1:]
    return temp


idx = df.columns.get_loc("D1")
df.iloc[:, idx:] = df.iloc[:, idx:].map(camelCaseString) #APPLICA A TUTTE LE D1-D11

DC = Namespace("http://purl.org/dc/elements/1.1/")
count=0
for s, p, o in g.triples((None, RDFS.subClassOf, URIRef("http://fairness_ontology.com/ScientificPaper"))):
    #print(f"{s} is an ScientificPaper")
    for s1, p1, o1 in g.triples((None, RDF.type, s)):
        title = g.value(s1, DC.title)
        matches = df[df["Paper"].str.contains(title, case=False, na=False, regex=False)]
        if not matches.empty:
            row = matches.iloc[0]
            #print(f"row: {row}")
            cols = df.columns[df.columns.get_loc("D1"):] # get all columns from D1 onward

            for col in cols:
                d1 = row[col].strip()
                #d1 = matches["D1"].iloc[0].strip()     #TODO : DA FARE PER TUTTE LE D1-D11
                if d1 not in ["", "~", "no experiment"]:
                    d1URIRef=URIRef("http://fairness_ontology.com/papers/"+d1) 
                    #print(d1URIRef)
                    g.add((d1URIRef,RDF.type,URIRef("http://fairness_ontology.com/Dataset")))
                    g.add((s1,URIRef("http://fairness_ontology.com/testedOn"),d1URIRef))
                    count+=1
g.serialize(destination="./outfiles/papers_upd1.ttl") 
#NOW DATASETS FROM HORT PAPER ARE LOADED IN THE ONTOLOGY AND CONNECTED TO THE PAPERS!
print(f"{count} testedOn relation imported!")  


848 testedOn relation imported!


In [ ]:
#POPULATE Fairness-Metrics.xlsx WITH NEWLY FOUND testedOn RELATIONS

#load the content of the Datasets and ApplicationDomain excel sheet
df = pd.read_excel("../Fairness-Metrics.xlsx", sheet_name="Datasets and ApplicationDomain",header=0, dtype=str)
df.fillna('', inplace=True)

df['Dataset']=df['Dataset'].apply(lambda x: camelCaseString(x))
##df.head()
for idx, row in enumerate(df.itertuples(index=False)):
    #datasetAlreadyInOnt=0
    datasetURI=URIRef("http://fairness_ontology.com/papers/"+row.Dataset)
    #se il dataset sull'excel è già nella ontologia, aggiungi tutti i paper su cui è testato (presi da HORT)
    for s, p, o in g.triples((datasetURI, RDF.type, URIRef("http://fairness_ontology.com/Dataset"))) :
        for s1, p1, o1 in g.triples((None,URIRef("http://fairness_ontology.com/testedOn"),datasetURI)) :
            paperInOnt=split_uri(s1)[1]
            #print(f"In the ontology: Found {row.Dataset} used by {paperInOnt} for testing") 
            if paperInOnt in row.ScientificPaper :
                #print("Relation already in Excel ")
                pass
            else:
                #print("Not in the excel")
                df.loc[idx, "ScientificPaper"] = (str(row.ScientificPaper) + "," + paperInOnt)
        #datasetAlreadyInOnt+=1
    #if (datasetAlreadyInOnt==0):
        #the dataset in the excel is not in the ontology ---> NO dovevo iterare sui dataset nell'ontologia!!!!
    #print(row.Dataset, row.ScientificPaper)


df.to_excel("output.xlsx", index=False)
df.head(10)

,Dataset,ApplicationDomain,Description,ScientificPaper,link
0,Amazon,RetailAndECommerce,Online shopping dataset containing product rev...,"2019_DELDJOO,2020_FU,2021_LI,2021_LIb,2021_WUb...",
1,CtripFlight,RetailAndECommerce,Flight dataset from an online travel company :...,"2021_WUb,2022_LIc",
2,Flixster,RetailAndECommerce,Movies and users dataset: classical movie reco...,2018_KAMISHIMA,
3,GoogleLocal,"RetailAndECommerce,PublicServices",Reviews about local businesses from Google Maps.,"2021_Wub,2020_PATRO",
4,Insurance,FinanceAndInsurance,Kaggle dataset with the goal of recommending i...,2021_Lia,
5,LastFM1K,RetailAndECommerce,music recommendation dataset,2018_EKSTRAND,
6,LastFM360K,RetailAndECommerce,music recommendation dataset,"2018_EKSTRAND,2021_MANSOURY,2020_PATRO,2021_WU",
7,ModCloth,RetailAndECommerce,women’s clothing website,2020_WAN,
8,Movielens100K,RetailAndECommerce,Movies and users dataset,"2021_GE,2018_LEONHARDT,2020_LIU,2018_SURER,201...",
9,Movielens1M,RetailAndECommerce,Movies and users dataset,"2018_KAMISHIMA,2021_ISLAM,2021_LI,2021_MANSOUR...",


In [ ]:
#POPULATE Fairness-Metrics.xlsx WITH NEWLY FOUND DATASETS IN THE ONTOLOGY AND THEIR testedOn RELATIONS
#HENCE, ITERATE ON THE ONTOLOGY'S DATASETS AND UPDATES THE EXCEL FILE WITH MISSING ONES

def datasetToSkip(ds):
    if ds.strip() in ["", "NoExperiment", "Synthetic","SemiSynthetic"]:
        return True
    else:
        return False


for s, p, o in g.triples((None,RDF.type,URIRef("http://fairness_ontology.com/Dataset"))):
    dataset=split_uri(s)[1]
    if not datasetToSkip(dataset):
        if not (df["Dataset"] == dataset).any():
            testedOn=""
            for s1,p1,o1 in g.triples((None,URIRef("http://fairness_ontology.com/testedOn"),s)) :
                testedOn+=str(split_uri(s1)[1]+",")
            if len(testedOn)!=0 :
                testedOn=testedOn[:-1] #remove last ","
            print(f"Found new one: {testedOn}. TestedOn {dataset}")
            new_row = {"Dataset": dataset, "ScientificPaper": testedOn, "ApplicationDomain": "", "Description": "", "link": ""}
            # Add row at the next index
            df.loc[len(df)] = new_row

df.to_excel("outfiles/output.xlsx", index=False)


Found new one: CreditApproval. TestedOn 2013_FUKUCHI,2015_FUKUCHIa
Found new one: Housing. TestedOn 2013_FUKUCHI,2015_FUKUCHIa
Found new one: Diabetes. TestedOn 2015_EDWARDS,2018_RAFF
Found new one: DarkReactions. TestedOn 2018_ADLER
Found new one: Internal. TestedOn 2018_GUPTA
Found new one: Adience. TestedOn 2019_ABUSITTA
Found new one: IBMHRAnalyticsEmployeeAttrition. TestedOn 2020_GARCIADEALFORD
Found new one: FLC. TestedOn 2020_KEYA
Found new one: UCINursery. TestedOn 2020_ROMANO
Found new one: CERISSDA. TestedOn 2021_LOHIA
Found new one: HospitalReadmission. TestedOn 2021_PETROVIC,2022_PETROVIC
Found new one: Salary. TestedOn 2021_VERMA,2022_WANGe
Found new one: ENEM. TestedOn 2022_ALGHAMDI
Found new one: ADS. TestedOn 2022_PENTYALA
Found new one: ML1M. TestedOn 2022_PENTYALA
Found new one: Multimorbidity. TestedOn 2022_SEKER
Found new one: NewOrleans. TestedOn 2022_SHORT
Found new one: CardioEicu. TestedOn 2022_SURIYAKUMAR
Found new one: Apnea. TestedOn 2022_SURIYAKUMAR,2019_UST